# Migrate to Apache Iceberg
## In-place Migration of Delta Lake Tables using Iceberg add_files procedure

### Configure Spark session with proper Delta Lake and Iceberg settings

In [ ]:
%%configure -f
{   
    "conf":{
        "numExecutors": 1, 
        "executorMemory": "14G",
        "executorCores": 4, 
        "spark.dynamicAllocation.enabled": "false",
        "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension",
        "spark.sql.catalog.iceberg_catalog": "org.apache.iceberg.spark.SparkCatalog",
        "spark.sql.catalog.iceberg_catalog.type": "glue",
        "spark.hadoop.hive.metastore.client.factory.class": "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory",
        "spark.jars": "/usr/share/aws/delta/lib/delta-spark.jar,/usr/share/aws/delta/lib/delta-storage.jar",
        "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    }
}

### Configure variables

In [ ]:
delta_table_name = "delta_table_test_add"
iceberg_table_name ="delta_table_test_add_iceberg"
delta_table_name_del_vect = "delta_table_del_vect_test_add"
iceberg_table_name_del_vect="delta_table_del_vect_test_add_iceberg"
bucket_name="mybucket"
db_name="mydb"

### 1. Delta Lake Table (No Deletion Vectors)
#### Delta Lake table creation and population

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{delta_table_name} (
  id INT,
  country STRING,
  name STRING
)
USING DELTA
PARTITIONED BY (country)
LOCATION 's3://{bucket_name}/{db_name}/{delta_table_name}/'
""")

In [ ]:
spark.sql(f"""
INSERT INTO {db_name}.{delta_table_name} VALUES 
  (1, 'USA','John Smith'),
  (2, 'USA','Mary Johnson'),
  (3, 'Italy', 'Marco Rossi'),
  (4, 'Italy', 'Sofia Bianchi');
""")

In [ ]:
spark.sql(f"""
UPDATE {db_name}.{delta_table_name}
SET name = 
  CASE 
    WHEN id = 1 THEN 'John2'
    WHEN id = 4 THEN 'Sofia2'
    ELSE name
  END
WHERE id IN (1, 4)
""")

#### Migration
1. Create an Iceberg table with the same schema

In [ ]:
spark.sql(f"""
CREATE TABLE iceberg_catalog.{db_name}.{iceberg_table_name} (
  id INT,
  country STRING,
  name STRING
)
USING iceberg
PARTITIONED BY (country)
LOCATION 's3://{bucket_name}/{db_name}/{iceberg_table_name}/'
""")

2. Run VACUUM operation to keep only the latest files versions

In [ ]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

In [ ]:
spark.sql(f"""
  VACUUM {db_name}.{delta_table_name} RETAIN 0 HOURS
""").show(truncate=False)

3. Use the add_files procedure to do in-place migration

In [ ]:
spark.sql(f""" 
CALL iceberg_catalog.system.add_files( 
source_table => '`parquet`.`s3://{bucket_name}/{db_name}/{delta_table_name}/`',  
table => 'iceberg_catalog.{db_name}.{iceberg_table_name}'
)
""").show(truncate=False)

4. Confirm the 2 tables are identical

In [ ]:
spark.sql(f"SELECT * FROM {db_name}.{delta_table_name} ORDER BY id").show(truncate=False)

In [ ]:
spark.sql(f"SELECT * FROM iceberg_catalog.{db_name}.{iceberg_table_name} ORDER BY id").show(truncate=False)

### 2. Delta Lake Table (Deletion Vectors)
#### Delta Lake table creation and population

In [ ]:
spark.sql(f"""
CREATE TABLE {db_name}.{delta_table_name_del_vect} (
  id INT,
  country STRING,
  name STRING
)
USING DELTA
PARTITIONED BY (country)
TBLPROPERTIES (
  'delta.enableDeletionVectors' = true
)
LOCATION 's3://{bucket_name}/{db_name}/{delta_table_name_del_vect}/'
""")

In [ ]:
spark.sql(f"""
INSERT INTO {db_name}.{delta_table_name_del_vect} VALUES 
  (1, 'USA','John Smith'),
  (2, 'USA','Mary Johnson'),
  (3, 'Italy', 'Marco Rossi'),
  (4, 'Italy', 'Sofia Bianchi');
""")

In [ ]:
spark.sql(f"""
UPDATE {db_name}.{delta_table_name_del_vect}
SET name = 
  CASE 
    WHEN id = 1 THEN 'John2'
    WHEN id = 4 THEN 'Sofia2'
    ELSE name
  END
WHERE id IN (1, 4)
""")

#### Migration
1. Create an Iceberg table with the same schema

In [ ]:
spark.sql(f"""
CREATE TABLE iceberg_catalog.{db_name}.{iceberg_table_name_del_vect} (
  id INT,
  country STRING,
  name STRING
)
USING iceberg
PARTITIONED BY (country)
LOCATION 's3://{bucket_name}/{db_name}/{iceberg_table_name_del_vect}/'
""")

2. Run REORG to compact files with deletion vectors

In [ ]:
df=spark.sql(f"""
  REORG TABLE {db_name}.{delta_table_name_del_vect}
  APPLY (PURGE)
""")

In [ ]:
df.select("metrics.*").show(truncate=False)

3. Run VACUUM operation to keep only the latest files versions

In [ ]:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

In [ ]:
spark.sql(f"""
  VACUUM {db_name}.{delta_table_name_del_vect} RETAIN 0 HOURS
""").show(truncate=False)

4. Use the add_files procedure to do in-place migration

In [ ]:
spark.sql(f""" 
CALL iceberg_catalog.system.add_files( 
source_table => '`parquet`.`s3://{bucket_name}/{db_name}/{delta_table_name_del_vect}/`',  
table => 'iceberg_catalog.{db_name}.{iceberg_table_name_del_vect}'
)
""").show(truncate=False)

5. Confirm the 2 tables are identical

In [ ]:
spark.sql(f"SELECT * FROM {db_name}.{delta_table_name_del_vect} ORDER BY id").show(truncate=False)

In [ ]:
spark.sql(f"SELECT * FROM iceberg_catalog.{db_name}.{iceberg_table_name_del_vect} ORDER BY id").show(truncate=False)